# DLAV Phase 3

Clean Phase 3 launcher notebook for sim-to-real generalization.
The planner uses only the front camera and `sdc_history_feature`, and predicts 60 future XY positions.


In [ ]:
from pathlib import Path

# ==================================
# Edit this cell before running
# ==================================
# Primary outputs are saved under outputs/runs/phase3/<timestamp>_<run_name>/.

# --- Colab and storage behavior ---
REPO_URL = 'https://github.com/math707/dlav-project.git'
COLAB_PROJECT_DIR = Path('/content/dlav-project')
MOUNT_DRIVE_IN_COLAB = True
COLAB_DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/dlav-project-runs')
DOWNLOAD_DATA_IF_MISSING = True
SYNC_RUN_TO_DRIVE = True
PHASE_NAME = 'phase3'

# --- Experiment identity and core training settings ---
RUN_NAME = None
MODEL_NAME = 'phase3_resnet18'
NUM_EPOCHS = 120
BATCH_SIZE = 32
TEST_BATCH_SIZE = 250
LR = 7e-4
LEARNING_RATE_NAME = 'manual'
WEIGHT_DECAY = 1e-4
REAL_TRAIN_COUNT = 500
SEED = 42

# --- Optional two-stage training ---
USE_TWO_STAGE_TRAINING = True
REAL_TRAIN_COUNT_STAGE1 = 700
STAGE1_LR = 5e-4
STAGE1_AUGMENTATION_STRENGTH = 0.6
STAGE1_NUM_EPOCHS = 100
STAGE1_EARLY_STOPPING_PATIENCE = 18
STAGE2_LR = 1e-4
STAGE2_AUGMENTATION_STRENGTH = 0.0
STAGE2_NUM_EPOCHS = 20
STAGE2_EARLY_STOPPING_PATIENCE = 8

# --- Submission-size behavior ---
# Keep this as None to use len(public_test_files) automatically.
EXPECTED_PUBLIC_TEST_SAMPLES = None

# --- Augmentation and backbone behavior ---
USE_AUGMENTATION = True
AUGMENTATION_STRENGTH = 0.6
PRETRAINED = True
BACKBONE_LEARNING_RATE = None
BACKBONE_LR_SCALE = 0.1
BACKBONE_WARMUP_EPOCHS = 3

# --- Scheduler and early stopping ---
USE_LR_SCHEDULER = True
SCHEDULER_NAME = 'plateau'
SCHEDULER_METRIC = 'val_ADE'
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 6
SCHEDULER_MIN_LR = 1e-5
EARLY_STOPPING_PATIENCE = 25
EARLY_STOPPING_MIN_DELTA = 1e-3

# --- Inference and output behavior ---
RELOAD_BEST_CHECKPOINT_FOR_INFERENCE = True
TOP_K_CHECKPOINTS = 5
GENERATE_LAST_CHECKPOINT_SUBMISSION = True
GENERATE_TOP_K_SUBMISSIONS = True
GENERATE_ENSEMBLE_SUBMISSION = True
ENSEMBLE_CHECKPOINT_LIMIT = 3

ACTIVE_REAL_TRAIN_COUNT = REAL_TRAIN_COUNT_STAGE1 if USE_TWO_STAGE_TRAINING else REAL_TRAIN_COUNT
ACTIVE_AUGMENTATION_STRENGTH = (
    STAGE1_AUGMENTATION_STRENGTH if USE_TWO_STAGE_TRAINING else AUGMENTATION_STRENGTH
)
ACTIVE_LR = STAGE1_LR if USE_TWO_STAGE_TRAINING else LR
ACTIVE_NUM_EPOCHS = (
    STAGE1_NUM_EPOCHS + STAGE2_NUM_EPOCHS if USE_TWO_STAGE_TRAINING else NUM_EPOCHS
)
LEARNING_RATE_OPTIONS = (
    {
        'one_stage': LR,
        'stage1': STAGE1_LR,
        'stage2': STAGE2_LR,
    }
    if USE_TWO_STAGE_TRAINING
    else {LEARNING_RATE_NAME: LR}
)

EXPERIMENT_NAME = f'{MODEL_NAME}_realmix{ACTIVE_REAL_TRAIN_COUNT}'
if USE_TWO_STAGE_TRAINING:
    EXPERIMENT_NAME += '_2stage'
if PRETRAINED:
    EXPERIMENT_NAME += '_pretrained'
if USE_AUGMENTATION and ACTIVE_AUGMENTATION_STRENGTH > 0:
    EXPERIMENT_NAME += f'_aug{ACTIVE_AUGMENTATION_STRENGTH:g}'
if USE_LR_SCHEDULER:
    EXPERIMENT_NAME += f'_{SCHEDULER_NAME}'
if WEIGHT_DECAY > 0:
    EXPERIMENT_NAME += f'_wd{WEIGHT_DECAY:g}'


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def _is_running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def _find_project_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    return None


def _bootstrap_project_root() -> tuple[Path, bool]:
    in_colab = _is_running_in_colab()
    project_root = _find_project_root(Path.cwd())

    if in_colab:
        if project_root is None:
            git_dir = COLAB_PROJECT_DIR / '.git'
            if git_dir.is_dir():
                print(f'Updating repository in {COLAB_PROJECT_DIR}...')
                subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
                project_root = COLAB_PROJECT_DIR
            elif COLAB_PROJECT_DIR.exists():
                if (COLAB_PROJECT_DIR / 'src').is_dir() and (COLAB_PROJECT_DIR / 'notebooks').is_dir():
                    print(f'Using existing project directory in {COLAB_PROJECT_DIR}...')
                    project_root = COLAB_PROJECT_DIR
                else:
                    raise FileExistsError(
                        f'{COLAB_PROJECT_DIR} exists but is not a recognized project root.'
                    )
            else:
                print(f'Cloning repository into {COLAB_PROJECT_DIR}...')
                subprocess.check_call(['git', 'clone', REPO_URL, str(COLAB_PROJECT_DIR)])
                project_root = COLAB_PROJECT_DIR
        elif project_root == COLAB_PROJECT_DIR and (COLAB_PROJECT_DIR / '.git').is_dir():
            print(f'Updating repository in {COLAB_PROJECT_DIR}...')
            subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
    elif project_root is None:
        raise FileNotFoundError(
            'Could not find the project root. Open the notebook from inside the repository.'
        )

    os.chdir(project_root)
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    return project_root.resolve(), in_colab


PROJECT_ROOT, IN_COLAB = _bootstrap_project_root()

from src.shared.project_setup import prepare_project_context

PROJECT = prepare_project_context(
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
    mount_drive_in_colab=MOUNT_DRIVE_IN_COLAB,
    drive_runs_root=COLAB_DRIVE_RUNS_ROOT,
    phase_name=PHASE_NAME,
)

PROJECT_ROOT = PROJECT.project_root
DATA_DIR = PROJECT.data_dir
TRAIN_DIR = DATA_DIR / 'train'
REAL_DIR = DATA_DIR / 'val_real'
TEST_PUBLIC_REAL_DIR = DATA_DIR / 'test_public_real'
RUNS_DIR = PROJECT.runs_dir
CHECKPOINT_DIR = PROJECT.checkpoints_dir
SUBMISSION_DIR = PROJECT.submissions_dir
LEGACY_CHECKPOINT_PATH = PROJECT.legacy_checkpoint_path
LEGACY_SUBMISSION_PATH = PROJECT.submissions_dir / 'submission_phase3.csv'

print(f'Environment: {"Google Colab" if PROJECT.in_colab else "Local"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Data directory: {DATA_DIR}')
print(f'Run directory root: {RUNS_DIR}')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Submission directory: {SUBMISSION_DIR}')
print(f'Public test directory: {TEST_PUBLIC_REAL_DIR}')


In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.phase3 import (
    DrivingDataset,
    PHASE3_DATASET_SPECS,
    build_model,
    build_phase3_splits,
    build_train_augmentations,
    generate_ensemble_submission,
    generate_submission,
    list_test_public_real_files,
    load_models_from_checkpoints,
    train,
    validate,
)
from src.shared.data_utils import ensure_all_datasets, has_pkl_files
from src.shared.logger import Logger
from src.shared.run_utils import (
    build_initial_run_metrics,
    copy_artifact_to_destination,
    create_run_context,
    save_metrics,
    sync_run_to_drive,
    write_summary,
)
from src.shared.training_setup import build_optimizer, build_scheduler


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 2 if (PROJECT.in_colab or os.name != 'nt') else 0
PIN_MEMORY = DEVICE.type == 'cuda'

if DOWNLOAD_DATA_IF_MISSING:
    ensure_all_datasets(
        data_dir=DATA_DIR,
        in_colab=PROJECT.in_colab,
        dataset_specs=PHASE3_DATASET_SPECS,
        project_root=PROJECT_ROOT,
    )
else:
    expected_dirs = {
        'train': TRAIN_DIR,
        'val_real': REAL_DIR,
        'test_public_real': TEST_PUBLIC_REAL_DIR,
    }
    missing_splits = [name for name, path in expected_dirs.items() if not has_pkl_files(path)]
    if missing_splits:
        missing_str = ', '.join(missing_splits)
        raise FileNotFoundError(
            f'Missing extracted dataset folders for: {missing_str}. '
            'Either enable DOWNLOAD_DATA_IF_MISSING or place the files manually under data/.'
        )

print(f'Device: {DEVICE}')
print(f'num_workers: {NUM_WORKERS}')
print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Model: {MODEL_NAME}')
print(f'Pretrained backbone: {PRETRAINED}')
print(f'Two-stage training: {USE_TWO_STAGE_TRAINING}')
print(f'Real train count: {ACTIVE_REAL_TRAIN_COUNT}')
if USE_TWO_STAGE_TRAINING:
    print(
        f'Stage 1 | lr={STAGE1_LR} | aug={STAGE1_AUGMENTATION_STRENGTH} | '
        f'epochs={STAGE1_NUM_EPOCHS} | early_stopping={STAGE1_EARLY_STOPPING_PATIENCE}'
    )
    print(
        f'Stage 2 | lr={STAGE2_LR} | aug={STAGE2_AUGMENTATION_STRENGTH} | '
        f'epochs={STAGE2_NUM_EPOCHS} | early_stopping={STAGE2_EARLY_STOPPING_PATIENCE}'
    )
else:
    print(f'Use augmentation: {USE_AUGMENTATION} | strength={AUGMENTATION_STRENGTH}')
    print(f'LR: {LR}')
print(f'Weight decay: {WEIGHT_DECAY}')


In [ ]:
RUN_CONTEXT = create_run_context(
    project_root=PROJECT_ROOT,
    in_colab=PROJECT.in_colab,
    run_name=RUN_NAME,
    default_run_name=EXPERIMENT_NAME,
    drive_root=PROJECT.drive_runs_root,
    phase_name=PHASE_NAME,
)

RUN_METRICS = build_initial_run_metrics(
    RUN_CONTEXT,
    model_name=MODEL_NAME,
    device=str(DEVICE),
    batch_size=BATCH_SIZE,
    learning_rate_name=LEARNING_RATE_NAME,
    learning_rate_options=LEARNING_RATE_OPTIONS,
    learning_rate=ACTIVE_LR,
    weight_decay=WEIGHT_DECAY,
    scheduler_enabled=USE_LR_SCHEDULER,
    scheduler_name=SCHEDULER_NAME,
    scheduler_metric=SCHEDULER_METRIC,
    num_epochs=ACTIVE_NUM_EPOCHS,
    legacy_checkpoint_path=LEGACY_CHECKPOINT_PATH,
    legacy_submission_path=LEGACY_SUBMISSION_PATH,
)
RUN_METRICS.update(
    {
        'pretrained_backbone': PRETRAINED,
        'training_mode': 'two_stage' if USE_TWO_STAGE_TRAINING else 'one_stage',
        'use_two_stage_training': USE_TWO_STAGE_TRAINING,
        'real_train_count': ACTIVE_REAL_TRAIN_COUNT,
        'seed': SEED,
        'use_augmentation': USE_AUGMENTATION,
        'augmentation_strength': ACTIVE_AUGMENTATION_STRENGTH,
        'backbone_lr_scale': BACKBONE_LR_SCALE,
        'backbone_warmup_epochs': BACKBONE_WARMUP_EPOCHS,
        'top_k_checkpoints': TOP_K_CHECKPOINTS,
        'ensemble_checkpoint_limit': ENSEMBLE_CHECKPOINT_LIMIT,
        'stage1_config': {
            'real_train_count': ACTIVE_REAL_TRAIN_COUNT,
            'learning_rate': STAGE1_LR if USE_TWO_STAGE_TRAINING else LR,
            'augmentation_strength': STAGE1_AUGMENTATION_STRENGTH if USE_TWO_STAGE_TRAINING else AUGMENTATION_STRENGTH,
            'num_epochs': STAGE1_NUM_EPOCHS if USE_TWO_STAGE_TRAINING else NUM_EPOCHS,
            'early_stopping_patience': STAGE1_EARLY_STOPPING_PATIENCE if USE_TWO_STAGE_TRAINING else EARLY_STOPPING_PATIENCE,
        },
        'stage2_config': (
            {
                'learning_rate': STAGE2_LR,
                'augmentation_strength': STAGE2_AUGMENTATION_STRENGTH,
                'num_epochs': STAGE2_NUM_EPOCHS,
                'early_stopping_patience': STAGE2_EARLY_STOPPING_PATIENCE,
            }
            if USE_TWO_STAGE_TRAINING
            else None
        ),
    }
)

save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Run name: {RUN_CONTEXT.run_name}')
print(f'Run directory: {RUN_CONTEXT.run_dir}')
print(f'Best checkpoint path: {RUN_CONTEXT.checkpoint_path}')
print(f'Submission path: {RUN_CONTEXT.submission_path}')


In [ ]:
SPLITS = build_phase3_splits(
    TRAIN_DIR,
    REAL_DIR,
    real_train_count=ACTIVE_REAL_TRAIN_COUNT,
    seed=SEED,
)
if not SPLITS['real_val']:
    raise ValueError('real_val split is empty. Reduce real_train_count to keep validation samples.')

stage1_augmentation_strength = (
    STAGE1_AUGMENTATION_STRENGTH if USE_TWO_STAGE_TRAINING else AUGMENTATION_STRENGTH
)
stage2_augmentation_strength = STAGE2_AUGMENTATION_STRENGTH if USE_TWO_STAGE_TRAINING else 0.0
stage1_train_transform = (
    build_train_augmentations(stage1_augmentation_strength)
    if USE_AUGMENTATION and stage1_augmentation_strength > 0
    else None
)
stage2_train_transform = (
    build_train_augmentations(stage2_augmentation_strength)
    if USE_TWO_STAGE_TRAINING and USE_AUGMENTATION and stage2_augmentation_strength > 0
    else None
)

stage1_train_dataset = DrivingDataset(
    SPLITS['mixed_train'],
    image_transform=stage1_train_transform,
    future_xy_only=True,
)
stage2_train_dataset = DrivingDataset(
    SPLITS['real_train'],
    image_transform=stage2_train_transform,
    future_xy_only=True,
)
train_dataset = stage1_train_dataset
val_dataset = DrivingDataset(SPLITS['real_val'], future_xy_only=True)
public_test_files = list_test_public_real_files(TEST_PUBLIC_REAL_DIR)
public_test_dataset = DrivingDataset(public_test_files, test=True)

stage1_train_loader = DataLoader(
    stage1_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
stage2_train_loader = DataLoader(
    stage2_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
train_loader = stage1_train_loader
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
public_test_loader = DataLoader(
    public_test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

sample = val_dataset[0]
RUN_METRICS.update(
    {
        'synthetic_train_count': len(SPLITS['synthetic_train']),
        'real_train_count_actual': len(SPLITS['real_train']),
        'real_val_count': len(SPLITS['real_val']),
        'mixed_train_count': len(SPLITS['mixed_train']),
        'stage1_train_count': len(stage1_train_dataset),
        'stage2_train_count': len(stage2_train_dataset),
        'stage1_augmentation_strength': stage1_augmentation_strength,
        'stage2_augmentation_strength': stage2_augmentation_strength,
        'public_test_count': len(public_test_dataset),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f"Synthetic train samples: {len(SPLITS['synthetic_train'])}")
print(f"Real train samples: {len(SPLITS['real_train'])}")
print(f"Real val samples: {len(SPLITS['real_val'])}")
print(f"Stage 1 train samples: {len(stage1_train_dataset)}")
print(f"Stage 2 real-only train samples: {len(stage2_train_dataset)}")
print(f"Public test samples: {len(public_test_dataset)}")
print(f"Public test directory: {TEST_PUBLIC_REAL_DIR}")
print(f"camera: {tuple(sample['camera'].shape)} | history: {tuple(sample['history'].shape)} | future: {tuple(sample['future'].shape)}")


In [ ]:
preview_count = min(4, len(val_dataset))
fig, axes = plt.subplots(2, preview_count, figsize=(4 * preview_count, 8))
if preview_count == 1:
    axes = np.array(axes).reshape(2, 1)

for index in range(preview_count):
    sample = val_dataset[index]
    camera = sample['camera'].permute(1, 2, 0).cpu().numpy().clip(0.0, 1.0)
    history = sample['history'].cpu().numpy()
    future = sample['future'].cpu().numpy()

    axes[0, index].imshow(camera)
    axes[0, index].set_title(f'Real sample {index}')
    axes[0, index].axis('off')

    axes[1, index].plot(history[:, 0], history[:, 1], 'o-', color='gold', label='Past')
    axes[1, index].plot(future[:, 0], future[:, 1], 'o-', color='green', label='Future')
    axes[1, index].axis('equal')
    axes[1, index].set_title('Trajectory')
    axes[1, index].legend()

plt.tight_layout()
plt.show()


In [ ]:
model = build_model(
    MODEL_NAME,
    pretrained_backbone=PRETRAINED,
)

class StageTaggedLogger:
    def __init__(self, base_logger, stage_name):
        self.base_logger = base_logger
        self.stage_name = stage_name

    def log(self, step=None, **metrics):
        self.base_logger.log(step=step, training_stage=self.stage_name, **metrics)


def build_stage_optimizer_and_scheduler(model, learning_rate):
    stage_optimizer = build_optimizer(
        model,
        learning_rate=learning_rate,
        weight_decay=WEIGHT_DECAY,
        backbone_learning_rate=BACKBONE_LEARNING_RATE,
        backbone_lr_scale=BACKBONE_LR_SCALE,
    )
    stage_scheduler = build_scheduler(
        stage_optimizer,
        enabled=USE_LR_SCHEDULER,
        name=SCHEDULER_NAME,
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=SCHEDULER_MIN_LR,
    )
    return stage_optimizer, stage_scheduler


def summarize_training_stage(
    stage_name,
    training_summary,
    *,
    learning_rate,
    augmentation_strength,
    num_epochs,
    early_stopping_patience,
    checkpoint_path,
    last_checkpoint_path,
    train_sample_count,
):
    final_metrics = training_summary.get('final', {})
    best_metrics = training_summary.get('best', {})
    return {
        'stage_name': stage_name,
        'learning_rate': learning_rate,
        'augmentation_strength': augmentation_strength,
        'num_epochs': num_epochs,
        'early_stopping_patience': early_stopping_patience,
        'train_sample_count': train_sample_count,
        'epochs_completed': training_summary.get('epochs_completed'),
        'train_loss_final': final_metrics.get('train_loss'),
        'val_loss_final': final_metrics.get('val_loss'),
        'val_ADE_final': final_metrics.get('val_ADE'),
        'val_FDE_final': final_metrics.get('val_FDE'),
        'best_val_ADE': best_metrics.get('val_ADE'),
        'best_val_ADE_epoch': best_metrics.get('epoch'),
        'best_val_FDE_at_best_ADE': best_metrics.get('val_FDE'),
        'best_val_loss_at_best_ADE': best_metrics.get('val_loss'),
        'best_checkpoint_path': best_metrics.get('checkpoint_path', str(checkpoint_path)),
        'last_checkpoint_path': str(last_checkpoint_path),
        'top_k_checkpoint_paths': training_summary.get('top_k_checkpoint_paths', []),
        'top_k_checkpoint_records': training_summary.get('top_k_checkpoint_records', []),
        'final_learning_rate': training_summary.get('final_learning_rate'),
        'final_learning_rates': training_summary.get('final_learning_rates', {}),
    }


def run_training_stage(
    stage_name,
    *,
    stage_train_loader,
    learning_rate,
    augmentation_strength,
    num_epochs,
    early_stopping_patience,
    checkpoint_path,
    last_checkpoint_path,
    backbone_warmup_epochs,
):
    stage_optimizer, stage_scheduler = build_stage_optimizer_and_scheduler(model, learning_rate)
    logger.log(
        training_stage=stage_name,
        stage_event='start',
        learning_rate=learning_rate,
        augmentation_strength=augmentation_strength,
        num_epochs=num_epochs,
        early_stopping_patience=early_stopping_patience,
        train_sample_count=len(stage_train_loader.dataset),
        checkpoint_path=str(checkpoint_path),
        last_checkpoint_path=str(last_checkpoint_path),
    )
    stage_training_summary = train(
        model,
        stage_train_loader,
        val_loader,
        stage_optimizer,
        StageTaggedLogger(logger, stage_name),
        num_epochs=num_epochs,
        scheduler=stage_scheduler,
        scheduler_metric=SCHEDULER_METRIC,
        best_checkpoint_path=checkpoint_path,
        last_checkpoint_path=last_checkpoint_path,
        top_k_checkpoints=TOP_K_CHECKPOINTS,
        early_stopping_patience=early_stopping_patience,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
        backbone_warmup_epochs=backbone_warmup_epochs,
    )
    stage_metrics = summarize_training_stage(
        stage_name,
        stage_training_summary,
        learning_rate=learning_rate,
        augmentation_strength=augmentation_strength,
        num_epochs=num_epochs,
        early_stopping_patience=early_stopping_patience,
        checkpoint_path=checkpoint_path,
        last_checkpoint_path=last_checkpoint_path,
        train_sample_count=len(stage_train_loader.dataset),
    )
    logger.log(
        training_stage=stage_name,
        stage_event='complete',
        best_val_ADE=stage_metrics.get('best_val_ADE'),
        best_checkpoint_path=stage_metrics.get('best_checkpoint_path'),
    )
    return stage_training_summary, stage_metrics


logger = Logger(log_path=RUN_CONTEXT.log_path)
LAST_CHECKPOINT_PATH = RUN_CONTEXT.run_dir / 'model_last.pth'
STAGE1_DIR = RUN_CONTEXT.run_dir / 'stage1'
STAGE1_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_CHECKPOINT_PATH = STAGE1_DIR / 'model.pth'
STAGE1_LAST_CHECKPOINT_PATH = STAGE1_DIR / 'model_last.pth'

stage1_checkpoint_path = STAGE1_CHECKPOINT_PATH if USE_TWO_STAGE_TRAINING else RUN_CONTEXT.checkpoint_path
stage1_last_checkpoint_path = STAGE1_LAST_CHECKPOINT_PATH if USE_TWO_STAGE_TRAINING else LAST_CHECKPOINT_PATH
stage1_learning_rate = STAGE1_LR if USE_TWO_STAGE_TRAINING else LR
stage1_num_epochs = STAGE1_NUM_EPOCHS if USE_TWO_STAGE_TRAINING else NUM_EPOCHS
stage1_early_stopping_patience = (
    STAGE1_EARLY_STOPPING_PATIENCE if USE_TWO_STAGE_TRAINING else EARLY_STOPPING_PATIENCE
)

STAGE1_TRAINING_SUMMARY, STAGE1_METRICS = run_training_stage(
    'stage1',
    stage_train_loader=stage1_train_loader,
    learning_rate=stage1_learning_rate,
    augmentation_strength=stage1_augmentation_strength,
    num_epochs=stage1_num_epochs,
    early_stopping_patience=stage1_early_stopping_patience,
    checkpoint_path=stage1_checkpoint_path,
    last_checkpoint_path=stage1_last_checkpoint_path,
    backbone_warmup_epochs=BACKBONE_WARMUP_EPOCHS,
)

STAGE2_TRAINING_SUMMARY = None
STAGE2_METRICS = None
TRAINING_SUMMARY = STAGE1_TRAINING_SUMMARY
FINAL_STAGE_METRICS = STAGE1_METRICS

if USE_TWO_STAGE_TRAINING:
    stage1_best_state_dict = torch.load(STAGE1_METRICS['best_checkpoint_path'], map_location=DEVICE)
    model.load_state_dict(stage1_best_state_dict)
    model = model.to(DEVICE)
    logger.log(
        training_stage='stage2',
        stage_event='reload_stage1_best',
        checkpoint_path=STAGE1_METRICS['best_checkpoint_path'],
    )
    STAGE2_TRAINING_SUMMARY, STAGE2_METRICS = run_training_stage(
        'stage2',
        stage_train_loader=stage2_train_loader,
        learning_rate=STAGE2_LR,
        augmentation_strength=stage2_augmentation_strength,
        num_epochs=STAGE2_NUM_EPOCHS,
        early_stopping_patience=STAGE2_EARLY_STOPPING_PATIENCE,
        checkpoint_path=RUN_CONTEXT.checkpoint_path,
        last_checkpoint_path=LAST_CHECKPOINT_PATH,
        backbone_warmup_epochs=0,
    )
    TRAINING_SUMMARY = STAGE2_TRAINING_SUMMARY
    FINAL_STAGE_METRICS = STAGE2_METRICS

RUN_METRICS.update(
    {
        'selected_training_stage': 'stage2' if USE_TWO_STAGE_TRAINING else 'stage1',
        'stage1_metrics': STAGE1_METRICS,
        'stage2_metrics': STAGE2_METRICS,
        'epochs_completed': FINAL_STAGE_METRICS.get('epochs_completed'),
        'train_loss_final': FINAL_STAGE_METRICS.get('train_loss_final'),
        'val_loss_final': FINAL_STAGE_METRICS.get('val_loss_final'),
        'val_ADE_final': FINAL_STAGE_METRICS.get('val_ADE_final'),
        'val_FDE_final': FINAL_STAGE_METRICS.get('val_FDE_final'),
        'best_val_ADE': FINAL_STAGE_METRICS.get('best_val_ADE'),
        'best_val_ADE_epoch': FINAL_STAGE_METRICS.get('best_val_ADE_epoch'),
        'best_val_FDE_at_best_ADE': FINAL_STAGE_METRICS.get('best_val_FDE_at_best_ADE'),
        'best_val_loss_at_best_ADE': FINAL_STAGE_METRICS.get('best_val_loss_at_best_ADE'),
        'best_checkpoint_path': FINAL_STAGE_METRICS.get('best_checkpoint_path'),
        'checkpoint_path': FINAL_STAGE_METRICS.get('best_checkpoint_path'),
        'last_checkpoint_path': FINAL_STAGE_METRICS.get('last_checkpoint_path'),
        'top_k_checkpoint_paths': FINAL_STAGE_METRICS.get('top_k_checkpoint_paths', []),
        'top_k_checkpoint_records': FINAL_STAGE_METRICS.get('top_k_checkpoint_records', []),
        'epoch_history': TRAINING_SUMMARY.get('history', []),
        'final_learning_rate': TRAINING_SUMMARY.get('final_learning_rate'),
        'final_learning_rates': TRAINING_SUMMARY.get('final_learning_rates', {}),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f"Stage 1 best val ADE: {STAGE1_METRICS['best_val_ADE']:.4f} at epoch {STAGE1_METRICS['best_val_ADE_epoch']}")
print(f"Stage 1 checkpoint: {STAGE1_METRICS.get('best_checkpoint_path')}")
if STAGE2_METRICS is not None:
    print(f"Stage 2 best val ADE: {STAGE2_METRICS['best_val_ADE']:.4f} at epoch {STAGE2_METRICS['best_val_ADE_epoch']}")
    print(f"Stage 2 checkpoint: {STAGE2_METRICS.get('best_checkpoint_path')}")
print(f"Final selected stage: {RUN_METRICS['selected_training_stage']}")
print(f"Best checkpoint: {RUN_METRICS.get('best_checkpoint_path')}")
if RUN_METRICS.get('top_k_checkpoint_paths'):
    print('Top-k checkpoints:', RUN_METRICS['top_k_checkpoint_paths'])
print(f'Run metrics updated: {RUN_CONTEXT.metrics_path}')
print(f'Run log: {RUN_CONTEXT.log_path}')


In [ ]:
copy_artifact_to_destination(RUN_CONTEXT.checkpoint_path, LEGACY_CHECKPOINT_PATH)

if RELOAD_BEST_CHECKPOINT_FOR_INFERENCE:
    best_state_dict = torch.load(RUN_CONTEXT.checkpoint_path, map_location=DEVICE)
    model.load_state_dict(best_state_dict)
    model = model.to(DEVICE)
    RUN_METRICS['inference_checkpoint_path'] = str(RUN_CONTEXT.checkpoint_path)
else:
    RUN_METRICS['inference_checkpoint_path'] = str(LAST_CHECKPOINT_PATH)

VALIDATION_METRICS = validate(model, val_loader, device=DEVICE)
RUN_METRICS.update(
    {
        'reloaded_val_ADE': VALIDATION_METRICS.get('val_ADE'),
        'reloaded_val_FDE': VALIDATION_METRICS.get('val_FDE'),
        'reloaded_val_loss': VALIDATION_METRICS.get('val_loss'),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Best model saved to {RUN_CONTEXT.checkpoint_path}')
print(f'Last-epoch model saved to {LAST_CHECKPOINT_PATH}')
print(f'Historical checkpoint copy saved to {LEGACY_CHECKPOINT_PATH}')
print(VALIDATION_METRICS)


In [ ]:
public_test_files = list_test_public_real_files(TEST_PUBLIC_REAL_DIR)
expected_public_test_samples = (
    len(public_test_files) if EXPECTED_PUBLIC_TEST_SAMPLES is None else EXPECTED_PUBLIC_TEST_SAMPLES
)
expected_submission_shape = (expected_public_test_samples, 121)
print(f'Public test files: {len(public_test_files)}')
print(f'Public test directory: {TEST_PUBLIC_REAL_DIR}')
print('First public test filenames:', [path.name for path in public_test_files[:5]])
print(f'Expected submission shape: {expected_submission_shape}')

public_test_dataset = DrivingDataset(public_test_files, test=True)
public_test_loader = DataLoader(
    public_test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

primary_checkpoint_path = RUN_METRICS.get('inference_checkpoint_path')
reload_primary_checkpoint = primary_checkpoint_path is not None

submission = generate_submission(
    model=model,
    output_path=RUN_CONTEXT.submission_path,
    device=DEVICE,
    data_loader=public_test_loader,
    checkpoint_path=primary_checkpoint_path,
    reload_checkpoint=reload_primary_checkpoint,
    legacy_output_path=LEGACY_SUBMISSION_PATH,
    copy_fn=copy_artifact_to_destination,
    expected_num_samples=expected_public_test_samples,
)

generated_submission_paths = {'selected': str(RUN_CONTEXT.submission_path)}

if GENERATE_LAST_CHECKPOINT_SUBMISSION:
    last_submission_path = RUN_CONTEXT.run_dir / 'submission_phase3_last.csv'
    generate_submission(
        model=model,
        output_path=last_submission_path,
        device=DEVICE,
        data_loader=public_test_loader,
        checkpoint_path=LAST_CHECKPOINT_PATH,
        reload_checkpoint=True,
        expected_num_samples=expected_public_test_samples,
    )
    generated_submission_paths['last'] = str(last_submission_path)

if GENERATE_TOP_K_SUBMISSIONS:
    for rank, record in enumerate(RUN_METRICS.get('top_k_checkpoint_records', []), start=1):
        topk_submission_path = RUN_CONTEXT.run_dir / f'submission_phase3_top{rank}.csv'
        generate_submission(
            model=model,
            output_path=topk_submission_path,
            device=DEVICE,
            data_loader=public_test_loader,
            checkpoint_path=record['checkpoint_path'],
            reload_checkpoint=True,
            expected_num_samples=expected_public_test_samples,
        )
        generated_submission_paths[f'top_{rank}'] = str(topk_submission_path)

if GENERATE_ENSEMBLE_SUBMISSION:
    ensemble_checkpoint_paths = []
    candidate_ensemble_paths = RUN_METRICS.get('top_k_checkpoint_paths', [])[:ENSEMBLE_CHECKPOINT_LIMIT]
    for checkpoint_path in candidate_ensemble_paths:
        if checkpoint_path not in ensemble_checkpoint_paths:
            ensemble_checkpoint_paths.append(checkpoint_path)
    if len(ensemble_checkpoint_paths) < 2:
        for checkpoint_path in [str(RUN_CONTEXT.checkpoint_path), str(LAST_CHECKPOINT_PATH)]:
            if checkpoint_path not in ensemble_checkpoint_paths:
                ensemble_checkpoint_paths.append(checkpoint_path)
    if len(ensemble_checkpoint_paths) >= 2:
        ensemble_models = load_models_from_checkpoints(
            ensemble_checkpoint_paths,
            model_factory=lambda: build_model(MODEL_NAME, pretrained_backbone=False),
            device=DEVICE,
        )
        ensemble_submission_path = RUN_CONTEXT.run_dir / 'submission_phase3_ensemble.csv'
        ensemble_submission = generate_ensemble_submission(
            models=ensemble_models,
            output_path=ensemble_submission_path,
            device=DEVICE,
            data_loader=public_test_loader,
            expected_num_samples=expected_public_test_samples,
        )
        generated_submission_paths['ensemble'] = str(ensemble_submission_path)
        RUN_METRICS['ensemble_checkpoint_paths'] = ensemble_checkpoint_paths
        RUN_METRICS['ensemble_submission_shape'] = list(ensemble_submission.shape)
    else:
        print('Skipping ensemble submission: need at least two unique checkpoint paths.')

RUN_METRICS['submission_path'] = str(RUN_CONTEXT.submission_path)
RUN_METRICS['submission_shape'] = list(submission.shape)
RUN_METRICS['expected_public_test_samples'] = expected_public_test_samples
RUN_METRICS['generated_submission_paths'] = generated_submission_paths
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

drive_backup_dir = sync_run_to_drive(RUN_CONTEXT) if SYNC_RUN_TO_DRIVE else None
if drive_backup_dir is not None:
    RUN_METRICS['drive_backup_enabled'] = True
    RUN_METRICS['drive_backup_path'] = str(drive_backup_dir)
    save_metrics(RUN_CONTEXT, RUN_METRICS)
    write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Submission saved to {RUN_CONTEXT.submission_path}')
print(f'Historical submission copy saved to {LEGACY_SUBMISSION_PATH}')
print('Generated submission artifacts:', generated_submission_paths)
if drive_backup_dir is not None:
    print(f'Run backup synchronized to {drive_backup_dir}')
print(f'Submission shape: {submission.shape}')
